#  `AIHUB 186.복지_분야_콜센터_상담데이터` 텍스트데이터 정보 획득

* 입력 폴더 위치: /home/data/data/aihub/186.복지_분야_콜센터_상담데이터/01.데이터
* 출력 폴더 위치: /home/data/expr/week2/data_prepare_aihub
* 출력 파일: orgtext.tsv
  * 탭으로 구분된 csv 파일 (tsv)
  * 열
      * file_label
      * category1
      * category2
      * category3
      * speaker_type
      * orgtext

### 입력폴더, 출력파일, 에러파일 정의

In [ ]:
input_dir = "/home/data/data/aihub/186.복지_분야_콜센터_상담데이터/01.데이터"
output_file = "/home/data/expr/week2/03-data_prepare_aihub/orgtext.tsv"
error_file = "/home/data/expr/week2/03-data_prepare_aihub/orgtext_errors.tsv"

### 라이브러리 로드

In [ ]:
from pathlib import Path
import json
import csv
from collections import Counter
from typing import Any

### 함수 정의

In [ ]:

def extract_orgtext_from_folder(
    input_dir: str,
    output_file: str = "orgtext.tsv",
    error_file: str = "orgtext_errors.tsv",
    include_relative_path: bool = False,
) -> None:
    """
    input_dir 이하의 모든 JSON 파일을 재귀적으로 탐색하여
    오류가 없는 JSON 파일만 TSV로 저장합니다.

    저장 열:
    - file_label
    - category1
    - category2
    - category3
    - speaker_type
    - orgtext
    """

    input_path = Path(input_dir).expanduser().resolve()
    output_path = Path(output_file).expanduser().resolve()
    error_path = Path(error_file).expanduser().resolve()

    if not input_path.exists():
        raise FileNotFoundError(f"입력 폴더가 없습니다: {input_path}")

    if not input_path.is_dir():
        raise NotADirectoryError(f"입력 경로가 폴더가 아닙니다: {input_path}")

    json_files = sorted(input_path.rglob("*.json"))

    success_count = 0
    row_count = 0
    error_count = 0
    excluded_file_count = 0
    file_labels = []

    output_path.parent.mkdir(parents=True, exist_ok=True)
    error_path.parent.mkdir(parents=True, exist_ok=True)

    with (
        output_path.open("w", encoding="utf-8-sig", newline="") as out_f,
        error_path.open("w", encoding="utf-8-sig", newline="") as err_f,
    ):
        writer = csv.writer(out_f, delimiter="\t")
        writer.writerow([
            "file_label",
            "category1",
            "category2",
            "category3",
            "speaker_type",
            "orgtext",
        ])

        error_writer = csv.writer(err_f, delimiter="\t")
        error_writer.writerow([
            "json_path",
            "error_type",
            "error_message",
        ])

        for json_path in json_files:
            pending_rows = []

            try:
                with json_path.open("r", encoding="utf-8-sig") as json_f:
                    data = json.load(json_f)

                if not isinstance(data, dict):
                    raise ValueError(
                        f"JSON 최상위 구조가 객체가 아닙니다: "
                        f"{type(data).__name__}"
                    )

                input_texts = data.get("inputText")

                if not isinstance(input_texts, list):
                    raise ValueError(
                        "inputText가 없거나 배열 형식이 아닙니다."
                    )

                if not input_texts:
                    raise ValueError("inputText 배열이 비어 있습니다.")

                info_list = data.get("info")

                if not isinstance(info_list, list) or not info_list:
                    raise ValueError(
                        "info가 없거나 비어 있거나 배열 형식이 아닙니다."
                    )

                first_info = info_list[0]

                if not isinstance(first_info, dict):
                    raise ValueError(
                        "info[0]이 객체 형식이 아닙니다."
                    )

                metadata = first_info.get("metadata")

                if not isinstance(metadata, dict):
                    raise ValueError(
                        "info[0].metadata가 없거나 객체 형식이 아닙니다."
                    )

                category1 = metadata.get("category1")
                category2 = metadata.get("category2")
                category3 = metadata.get("category3")
                speaker_type = metadata.get("speaker_type")

                required_metadata = {
                    "category1": category1,
                    "category2": category2,
                    "category3": category3,
                    "speaker_type": speaker_type,
                }

                for field_name, field_value in required_metadata.items():
                    if field_value is None:
                        raise ValueError(
                            f"metadata.{field_name} 값이 없습니다."
                        )

                    if not isinstance(field_value, str):
                        raise ValueError(
                            f"metadata.{field_name}가 문자열이 아닙니다: "
                            f"{type(field_value).__name__}"
                        )

                    if not field_value.strip():
                        raise ValueError(
                            f"metadata.{field_name} 값이 비어 있습니다."
                        )

                if include_relative_path:
                    file_label = str(json_path.relative_to(input_path))
                else:
                    file_label = json_path.stem

                for item_index, item in enumerate(input_texts):
                    if not isinstance(item, dict):
                        raise ValueError(
                            f"inputText[{item_index}]가 객체 형식이 아닙니다."
                        )

                    orgtext = item.get("orgtext")

                    if orgtext is None:
                        raise ValueError(
                            f"inputText[{item_index}].orgtext가 없습니다."
                        )

                    if not isinstance(orgtext, str):
                        raise ValueError(
                            f"inputText[{item_index}].orgtext가 "
                            f"문자열이 아닙니다: {type(orgtext).__name__}"
                        )

                    cleaned_text = (
                        orgtext
                        .replace("\t", " ")
                        .replace("\r", " ")
                        .replace("\n", " ")
                        .strip()
                    )

                    if not cleaned_text:
                        raise ValueError(
                            f"inputText[{item_index}].orgtext가 "
                            "빈 문자열입니다."
                        )

                    pending_rows.append([
                        file_label,
                        category1.strip(),
                        category2.strip(),
                        category3.strip(),
                        speaker_type.strip(),
                        cleaned_text,
                    ])

                if not pending_rows:
                    raise ValueError(
                        "저장할 수 있는 orgtext가 없습니다."
                    )

                # 파일 전체 검사가 끝난 뒤에만 최종 TSV에 저장
                writer.writerows(pending_rows)

                file_labels.append(file_label)
                success_count += 1
                row_count += len(pending_rows)

            except (
                json.JSONDecodeError,
                UnicodeDecodeError,
                OSError,
                ValueError,
            ) as e:
                excluded_file_count += 1
                error_count += 1

                print(f"[제외] {json_path}: {e}")

                error_writer.writerow([
                    str(json_path),
                    type(e).__name__,
                    str(e),
                ])

    label_counts = Counter(file_labels)

    duplicate_labels = {
        label: count
        for label, count in label_counts.items()
        if count > 1
    }

    print()
    print(f"입력 폴더: {input_path}")
    print(f"검색한 JSON 파일: {len(json_files):,}개")
    print(f"정상 저장 파일: {success_count:,}개")
    print(f"제외된 파일: {excluded_file_count:,}개")
    print(f"저장 행 수: {row_count:,}개")
    print(f"오류 파일 수: {error_count:,}개")
    print(f"출력 파일: {output_path}")
    print(f"오류 파일: {error_path}")

    if duplicate_labels:
        print(
            f"\n[중복 발견] "
            f"중복 file_label 수: {len(duplicate_labels):,}"
        )

        for label, count in sorted(duplicate_labels.items()):
            print(f"{label}\t{count}")
    else:
        print("\n[정상] 저장된 모든 file_label이 유일합니다.")


### 텍스트 데이터 추출 실행

In [ ]:
extract_orgtext_from_folder(
    input_dir=input_dir,
    output_file=output_file,
    error_file=error_file,
    include_relative_path=False,
)